In [114]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json

In [115]:
load_dotenv(override=True)

True

In [116]:
openai = OpenAI()
model_name = "gpt-5.2"

In [117]:
def show(text):
    Console(force_jupyter=True).print(text)

In [118]:
todos, completed = [], []

In [119]:
def get_todos_report():
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index+1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index+1}: [red]{todo}[/red]\n"
    Console(force_jupyter=True).print(result)
    return result

def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todos_report()

def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console(force_jupyter=True).print(completion_notes)
    return get_todos_report()

In [120]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
            },
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [121]:
complete_todo_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "type": "integer",
                "description": "The 1-based index of the todo to mark as complete"
            },
            "completion_notes": {
                "type": "string",
                "description": "Notes about how you completed the todo in rich console markup"
            }
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False
    }
}

In [122]:
tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": complete_todo_json}
]

In [123]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        if not tool:
            raise ValueError(f"Tool '{tool_name}' not found — check that the JSON schema name matches the Python function name")
        result = tool(**arguments)
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(
            model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none"
        )
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [124]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

user_message = """
Estimate the total cost of a road trip from San Francisco to New York.
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

In [125]:
todos, completed = [], []
loop(messages)

Tool called: create_todos


Todo #1: Assume a reasonable route distance and trip duration (miles, days).
Todo #2: Estimate fuel cost based on typical MPG and gas prices along the route.
Todo #3: Estimate lodging cost (number of hotel nights and nightly rate).
Todo #4: Estimate food cost per day for the trip duration.
Todo #5: Estimate tolls, parking, and incidentals (including NYC parking).
Todo #6: Sum all components and present a clear total with low/mid/high range.

Tool called: mark_complete


Assumed common I-80 style route SF→NYC: ~2,900 miles one-way. Typical driving pace ~500 mi/day ⇒ ~6 driving days, 
~5 hotel nights (sleeping at destination last day).

Todo #1: Assume a reasonable route distance and trip duration (miles, days).
Todo #2: Estimate fuel cost based on typical MPG and gas prices along the route.
Todo #3: Estimate lodging cost (number of hotel nights and nightly rate).
Todo #4: Estimate food cost per day for the trip duration.
Todo #5: Estimate tolls, parking, and incidentals (including NYC parking).
Todo #6: Sum all components and present a clear total with low/mid/high range.

...

Todo #1: Assume a reasonable route distance and trip duration (miles, days).
Todo #2: Estimate fuel cost based on typical MPG and gas prices along the route.
Todo #3: Estimate lodging cost (number of hotel nights and nightly rate).
Todo #4: Estimate food cost per day for the trip duration.
Todo #5: Estimate tolls, parking, and incidentals (including NYC parking).
Todo #6: Sum all components and present a clear total with low/mid/high range.

Tool called: mark_complete


Summed categories:
• Low: Fuel 406 + Lodging 550 + Food 240 + Tolls/parking/misc 140 = **$1,336**
• Mid: Fuel 464 + Lodging 750 + Food 360 + Tolls/parking/misc 260 = **$1,834**
• High: Fuel 522 + Lodging 1,100 + Food 540 + Tolls/parking/misc 470 = **$2,632**
Rounded for presentation.

Todo #1: Assume a reasonable route distance and trip duration (miles, days).
Todo #2: Estimate fuel cost based on typical MPG and gas prices along the route.
Todo #3: Estimate lodging cost (number of hotel nights and nightly rate).
Todo #4: Estimate food cost per day for the trip duration.
Todo #5: Estimate tolls, parking, and incidentals (including NYC parking).
Todo #6: Sum all components and present a clear total with low/mid/high range.

Assumptions (one-way SF → NYC)  
• Distance: ~2,900 miles  
• Pace: ~6 driving days (~500 mi/day)  
• Lodging: ~5 hotel nights (en route)  
• Vehicle: ~25 mpg

Cost estimate (one-way)

• Fuel (116 gal):  
  – Low ($3.50/gal): $406  
  – Mid ($4.00/gal): $464  
  – High ($4.50/gal): $522

• Lodging (5 nights):  
  – Low ($110/night): $550  
  – Mid ($150/night): $750  
  – High ($220/night): $1,100

• Food (6 days):  
  – Low ($40/day): $240  
  – Mid ($60/day): $360  
  – High ($90/day): $540

• Tolls + parking + incidentals (incl. ~1 night NYC parking):  
  – Low: $140  
  – Mid: $260  
  – High: $470

Total estimated trip cost (one-way)  
• Low: $406 + $550 + $240 + $140 = ~$1,340  
• Mid: $464 + $750 + $360 + $260 = ~$1,830  
• High: $522 + $1,100 + $540 + $470 = ~$2,630

Note: This excludes vehicle wear/maintenance and any extra nights spent in NYC (which can add substantially, 
especially for parking and lodging).